# Practice 5. Optimising a recipe: the least energy for the required strength

Industrial AI, KBTU. Goes with Lecture 9 and `inclass_work_9.ipynb`. **1.25 points.**

Data: UCI Concrete Compressive Strength, Kaggle copy
<https://www.kaggle.com/datasets/maajdl/yeh-concret-data>. The notebook downloads it by itself;
without internet it looks for `data/concrete/` in the course repository.

### The plant and the question

A precast concrete plant makes beams to order. Every batch is mixed from seven ingredients:
Portland cement; ground blast-furnace slag and fly ash, two industrial by-products that partly
replace cement; water; a superplasticizer, a chemical that lets the mix flow with less water;
coarse aggregate (gravel); and fine aggregate (sand). The customer requires a compressive
strength of at least **40 MPa at 28 days**.

The data are 1030 laboratory tests collected by I-Cheng Yeh (UCI Machine Learning Repository,
id 165, 1998). Each row is one mix in kg per m³ of concrete, the age of the test cube in days, and
the measured strength in MPa. The same mix was often tested at several ages.

A kilogram of cement takes far more energy to make than anything else in the mix, because the
kiln burns limestone at 1450 °C. Rounded values of the embodied energy, of the order given by
inventories such as ICE (University of Bath), in MJ per kg:

| cement | slag | fly ash | water | superplasticizer | coarse aggregate | fine aggregate |
|---|---|---|---|---|---|---|
| 4.6 | 1.6 | 0.1 | 0 | 18 | 0.08 | 0.08 |

Real numbers depend on the plant and the supplier; in this practice they are fixed.

Which mix reaches 40 MPa at 28 days with the least embodied energy? The plant cannot cast a
thousand trial batches. It has the 1030 tests, a surrogate model fitted to them, and an
optimiser. There is no simulator to check the answer against, so deciding how far the answer can
be trusted is part of the work.

### How the tasks are built

Each task says what to write: the functions, their arguments and what they return, and the
variables that hold your results. How you write them is up to you, as long as you use the method
the task names. Under your cell sits a cell of **tests**. Run it after your code: it either prints
`all tests passed` or stops at the first check that failed and says what is wrong. The tests check
what your code does on small cases with known answers, not how it is written.

These tests are public. When you hand in, the same tests run again together with hidden ones, on
other inputs and on your results, and the share of tests passed gives the points for the code.
Code that only fits the public cases loses those points.

Cells marked *Common ground* are written for you. Run them as they are: they fix the constants,
the surrogate models, the call counter and the drivers of Nelder–Mead, differential evolution and
SQP, so that everybody's numbers can be compared.

The methods are yours: grid search, random search, the Nelder–Mead step, the operators of
differential evolution, slopes by finite differences and the damped BFGS update. `scipy.optimize`
may be used to check a result, never to produce one. The one library call inside the given SQP is
`scipy.optimize.nnls`.

### What you hand in

Two files:

1. this notebook, run top to bottom, named `Surname_Name_practice_5.ipynb`;
2. `practice_5_answers.md` with your answers to the questions under each task, named
   `Surname_Name_practice_5_answers.md`.

Every answer needs the numbers your notebook printed. A number the notebook does not print scores
zero, and so does a number printed with nothing said about it.

| Task | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | total |
|---|---|---|---|---|---|---|---|---|---|
| code (tests) | 0.05 | 0.07 | 0.06 | 0.12 | 0.12 | 0.12 | 0.10 | 0.04 | 0.68 |
| answers | 0.05 | 0.08 | 0.04 | 0.08 | 0.08 | 0.08 | 0.10 | 0.06 | 0.57 |

### The rules

- **Every optimum is reported with its constraints.** An energy quoted without the predicted
  strength and the volume of the same mix scores zero in that answer.
- Every optimiser runs with a fixed seed, and the seed is printed next to its result.
- "The same budget" means the same number of calls to the objective, counted in code with
  `Counter`.
- Do not rename the functions and variables the tests use, and do not edit the test cells.
- Before handing in, restart the kernel and run all cells; the whole notebook runs in about five
  minutes on a laptop. Then check your answers against the new output.
- Figures need axis labels and a title that says what to look at.
- Stuck? `inclass_work_9.ipynb` does every step on a simulated dryer.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

import itertools
import scipy
import sklearn

BLUE, ORANGE, RED, GREEN, PURPLE, GREY = (
    "#2a6f8e", "#e07b39", "#c0392b", "#4a8f5a", "#9b3f9b", "#9aa0a6")

mpl.rcParams.update({
    "figure.dpi": 110,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#cccccc",
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e6e6e6",
    "grid.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.labelsize": 10,
    "font.size": 10,
    "font.family": ["Segoe UI", "DejaVu Sans", "sans-serif"],
    "legend.frameon": False,
    "lines.linewidth": 1.6,
})

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

KAGGLE = {
    'concrete': 'maajdl/yeh-concret-data',
}


def dataset_folder(name):
    """The folder with the files of a data set: the Kaggle copy (downloaded once, then cached),
    or data/<name>/ of the course repo if Kaggle is out of reach."""
    if KAGGLE[name]:
        try:
            import contextlib, io, logging, kagglehub
            logging.getLogger("kagglehub").setLevel(logging.ERROR)
            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                folder = Path(kagglehub.dataset_download(KAGGLE[name]))
            print(f"{name}: Kaggle {KAGGLE[name]}")
            return folder
        except Exception as exc:
            print(f"Kaggle unavailable ({type(exc).__name__}), looking for data/{name}/")
    for root in [Path.cwd(), *Path.cwd().parents]:
        if (root / "data" / name).is_dir():
            return root / "data" / name
    raise FileNotFoundError(f"no data/{name}/ here or above; run data/fetch.py in the course repo")

print(" | ".join(f"{m} {sys.modules[m].__version__}" for m in
                 ("numpy", "pandas", "matplotlib", "scipy", "sklearn", "torch") if m in sys.modules))

numpy 2.5.5 | pandas 3.1.1 | matplotlib 3.11.2 | scipy 1.18.1 | sklearn 1.9.1


In [2]:
raw = pd.read_csv(next(dataset_folder("concrete").rglob("Concrete_Data_Yeh.csv")))
print(raw.shape)
raw.head()

concrete: Kaggle maajdl/yeh-concret-data
(1030, 9)


,cement,slag,flyash,water,superplasticizer,coarseaggregate,fineaggregate,age,csMPa
0,540.000,0.000,0.000,162.000,2.500,"1,040.000",676.000,28,79.990
1,540.000,0.000,0.000,162.000,2.500,"1,055.000",676.000,28,61.890
2,332.500,142.500,0.000,228.000,0.000,932.000,594.000,270,40.270
3,332.500,142.500,0.000,228.000,0.000,932.000,594.000,365,41.050
4,198.600,132.400,0.000,192.000,0.000,978.400,825.500,360,44.300


---

## Common ground: the constants

Run this cell as it is. It names the seven ingredients in the order of the file and fixes, in the
same order, their embodied energy and density, then the requirement, the volume limits and the
penalty weight of Task 3.

In [3]:
MIX = ["cement", "slag", "flyash", "water", "superplasticizer",
       "coarseaggregate", "fineaggregate"]
# per ingredient, in the order of MIX: MJ per kg and kg per m3
ENERGY = np.array([4.6, 1.6, 0.1, 0.0, 18.0, 0.08, 0.08])
DENSITY = np.array([3150, 2900, 2300, 1000, 1200, 2700, 2650.0])
TARGET, AGE = 40.0, 28              # MPa, days
VOL_LO, VOL_HI = 0.96, 1.00     # m3 of ingredients per m3 of concrete; the rest is air
W = 1000.0                          # MJ per MPa: the penalty weight of Task 3

constants = pd.DataFrame({"energy, MJ/kg": ENERGY, "density, kg/m3": DENSITY}, index=MIX)
print(constants.T.to_string())

                  cement      slag    flyash     water  superplasticizer  coarseaggregate  fineaggregate
energy, MJ/kg      4.600     1.600     0.100     0.000            18.000            0.080          0.080
density, kg/m3 3,150.000 2,900.000 2,300.000 1,000.000         1,200.000        2,700.000      2,650.000


---

## Task 1. What is in the file  
*0.10 points: 0.05 for the code (tests), 0.05 for the answers*

Take inventory before you model anything. Rows with the same seven ingredients are one **mix**,
whatever the age of the test. With the masses $m_i$ in kg per m³, the energies $e_i$ (`ENERGY`) and
the densities $\rho_i$ (`DENSITY`):

| quantity | formula | unit |
|---|---|---|
| embodied energy | $E = \sum_i m_i e_i$ | MJ per m³ |
| volume of the ingredients | $V = \sum_i m_i / \rho_i$ | m³ |

The **baseline** is the best the plant already knows how to make: among the 28-day tests that
reached 40 MPa, the mix with the least energy. Every optimiser below has to beat it.

**Write**

- `n_duplicates`, `n_mixes`, `n_multi_age`: Python or numpy integers, the number of exact duplicate
  rows of `raw`, of distinct mixes, and of mixes tested at more than one age;
- `d28`: a `pd.DataFrame`, the rows of `raw` with `age` 28 without exact duplicates, with a new
  column `"energy"`, $E$ of each row;
- `baseline`: the row of `d28` (a `pd.Series`) with the least energy among those with `csMPa` at
  least `TARGET`.

In [ ]:
n_duplicates = ...
n_mixes = ...
n_multi_age = ...
d28 = ...
baseline = ...

In [ ]:
# Tests for Task 1 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_counts_are_integers():
    counts = {"n_duplicates": n_duplicates, "n_mixes": n_mixes, "n_multi_age": n_multi_age}
    for name, v in counts.items():
        assert isinstance(v, (int, np.integer)), \
            f"{name} must be an integer, got {type(v).__name__}"
    assert 0 < n_multi_age < n_mixes < len(raw), \
        "fewer mixes than rows, fewer repeated mixes than mixes"


test_counts_are_integers()


def test_d28_holds_the_28_day_tests():
    assert isinstance(d28, pd.DataFrame) and "energy" in d28, \
        "d28: a DataFrame with a column energy"
    assert (d28["age"] == 28).all(), "d28 holds only the tests at 28 days"
    assert not d28.duplicated().any(), "d28 has no exact duplicate rows"
    assert np.allclose(d28["energy"], d28[MIX].to_numpy() @ ENERGY), \
        "energy = sum of mass times MJ per kg"


test_d28_holds_the_28_day_tests()


def test_baseline_is_the_cheapest_strong_mix():
    assert isinstance(baseline, pd.Series), "baseline: one row of d28, a pd.Series"
    assert baseline["csMPa"] >= TARGET, "the baseline reached the target strength"
    strong = d28[d28["csMPa"] >= TARGET]
    assert np.isclose(baseline["energy"], strong["energy"].min()), \
        "no 28-day test of 40 MPa or more uses less energy"


test_baseline_is_the_cheapest_strong_mix()


print("all tests passed")

**Questions** (answer them in `practice_5_answers.md`, Task 1, with the numbers your code printed):

1. How many rows, exact duplicate rows and distinct mixes are there? How many mixes were tested at more than one age, and which ages occur?
2. What range does the volume of the ingredients take, and why is it not exactly 1 m³?
3. How many 28-day tests are there, and how many reach 40 MPa? Print the baseline with its ingredients, strength and energy, and plot strength against energy for the 28-day tests. What does the baseline do differently from a typical strong mix?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Common ground for Tasks 2–8

Run this cell as it is. It removes the exact duplicate rows and fixes the bounds of the search:
the smallest and the largest amount of each ingredient the lab ever used. It also gives three
model recipes:

- `make_poly()`, a quadratic polynomial with a light ridge penalty;
- `make_forest()`, a random forest of 200 trees;
- `make_mlp(seed)`, a network with two hidden layers of 64 `tanh` units.

All three take `features(mixes, age)`: the seven ingredients and $\log(\text{age})$. Concrete gains
strength fast in the first week and slowly after a month; on a log scale of age that growth is
much closer to a straight line.

In [7]:
concrete = raw.drop_duplicates().reset_index(drop=True)
# the bounds of the search: what the lab has tried
LO, HI = concrete[MIX].min().to_numpy(), concrete[MIX].max().to_numpy()
BOUNDS = list(zip(LO, HI))


def features(mixes, age):
    # Model inputs (n, 8): the seven ingredients and the log of the age in days.
    mixes = np.atleast_2d(mixes)
    age = np.broadcast_to(np.asarray(age, float), (len(mixes),))
    return np.column_stack([mixes, np.log(age)])


X_all = features(concrete[MIX].to_numpy(), concrete["age"].to_numpy())
y_all = concrete["csMPa"].to_numpy()

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import KFold, GroupKFold, cross_val_predict


def make_poly():
    return make_pipeline(StandardScaler(), PolynomialFeatures(2), Ridge(alpha=1.0))


def make_forest():
    return RandomForestRegressor(200, min_samples_leaf=2, random_state=0, n_jobs=1)


def make_mlp(seed=0):
    mlp = MLPRegressor(hidden_layer_sizes=(64, 64), activation="tanh", alpha=1e-3,
                       max_iter=4000, random_state=seed)
    return make_pipeline(StandardScaler(), mlp)


print(f"{len(concrete)} rows after removing exact duplicates; bounds of the search:")
print(pd.DataFrame({"low": LO, "high": HI}, index=MIX).T.to_string())

1005 rows after removing exact duplicates; bounds of the search:
      cement    slag  flyash   water  superplasticizer  coarseaggregate  fineaggregate
low  102.000   0.000   0.000 121.800             0.000          801.000        594.000
high 540.000 359.400 200.100 247.000            32.200        1,145.000        992.600


---

## Task 2. A surrogate for the strength, and a leak  
*0.15 points: 0.07 for the code (tests), 0.08 for the answers*

A **group** is all the rows of one mix: the same seven ingredients, tested at different
ages. `GroupKFold` keeps every group whole inside one fold, so a mix the model is scored on was
never seen in training at any age. `KFold` over the rows does not: the same mix tested at 7 and at
56 days can sit in training while its 28-day test is scored.

The **out-of-fold** RMSE: every row is predicted by the model fitted on the four folds that do not
hold it, $\text{RMSE} = \sqrt{\frac1n \sum_j (\hat y_j - y_j)^2}$ over all rows.
`cross_val_predict(model, X_all, y_all, cv=..., groups=...)` returns those predictions.

**Write**

- `mix_groups(df)`: a 1-D integer array of length `len(df)`, equal for rows with the same seven
  ingredients `MIX` and different otherwise;
- `cv_rmse`: a `pd.DataFrame` indexed by `"quadratic"`, `"forest"`, `"MLP"` (`make_poly()`,
  `make_forest()`, `make_mlp()`) with two columns of out-of-fold RMSE in MPa on `X_all`, `y_all`:
  `"rows"` with `KFold(5, shuffle=True, random_state=0)` and `"mixes"` with `GroupKFold(5)` and the
  groups `mix_groups(concrete)`. The MLP takes about half a minute for both splits.

In [ ]:
def mix_groups(df):
    ...


cv_rmse = ...

In [ ]:
# Tests for Task 2 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_groups_of_a_small_table():
    rows = [[300, 0, 0, 180, 0, 1000, 800, 7],
            [250, 50, 0, 180, 0, 1000, 800, 28],
            [300, 0, 0, 180, 0, 1000, 800, 28],
            [300, 0, 0, 170, 0, 1000, 800, 28]]
    demo = pd.DataFrame(rows, columns=MIX + ["age"])
    g = np.asarray(mix_groups(demo))
    assert g.shape == (4,), f"one id per row, got shape {g.shape}"
    assert g[0] == g[2], "rows 0 and 2 are the same mix tested at two ages"
    assert len(set(g.tolist())) == 3, "rows 0/2, 1 and 3 are three different mixes"


test_groups_of_a_small_table()


def test_groups_of_the_data():
    g = np.asarray(mix_groups(concrete))
    assert len(g) == len(concrete), "one id per row of concrete"
    n_distinct = len(concrete.drop_duplicates(subset=MIX))
    assert len(np.unique(g)) == n_distinct, "one id per distinct mix"


test_groups_of_the_data()


def test_cv_table_shape():
    assert isinstance(cv_rmse, pd.DataFrame), "cv_rmse: a DataFrame"
    assert list(cv_rmse.index) == ["quadratic", "forest", "MLP"], \
        "index: quadratic, forest, MLP"
    assert {"rows", "mixes"} <= set(cv_rmse.columns), "columns rows and mixes"
    rmse = cv_rmse[["rows", "mixes"]]
    assert ((rmse > 2) & (rmse < 15)).all().all(), "RMSE in MPa"


test_cv_table_shape()


print("all tests passed")

**Questions** (answer them in `practice_5_answers.md`, Task 2, with the numbers your code printed):

1. Print the 3 × 2 table of cross-validated RMSE, with the difference between the two splits for each model.
2. Which of the two splits is honest for *this* use of the model, and why? The optimiser will ask the model about mixes nobody has ever made.
3. How big is the leak for each model? Which model would you pick as the surrogate, and what error in MPa should you expect from it on a new mix?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Common ground for Tasks 3–8

Run this cell as it is. It fits the MLP and the forest on all the tests and gives one helper,
`strength(model, mixes)`, the predicted strength at 28 days. It also prints the MLP's error on
unseen mixes, `RMSE_MLP`, which Task 8 needs.

In [11]:
GROUPS = concrete.groupby(MIX, sort=False).ngroup().to_numpy()   # one id per distinct mix
oof = cross_val_predict(make_mlp(), X_all, y_all, cv=GroupKFold(5), groups=GROUPS)
RMSE_MLP = float(np.sqrt(np.mean((oof - y_all) ** 2)))

mlp = make_mlp().fit(X_all, y_all)
forest = make_forest().fit(X_all, y_all)


def strength(model, mixes):
    # Predicted 28-day strength, MPa, of one mix (7,) or of many (n, 7);
    # always an array (n,).
    return model.predict(features(mixes, AGE))


print(f"surrogates fitted on all {len(y_all)} tests; "
      f"MLP error on unseen mixes: RMSE {RMSE_MLP:.2f} MPa")

surrogates fitted on all 1005 tests; MLP error on unseen mixes: RMSE 5.92 MPa


---

## Task 3. The objective and the constraints  
*0.10 points: 0.06 for the code (tests), 0.04 for the answers*

The decision variables are the seven amounts $\mathbf{m} = (m_1, \dots, m_7)$, kg per m³, and the
problem is

$$\min_{\mathbf{m}} E(\mathbf{m}) \quad\text{subject to}\quad \hat f(\mathbf{m}) \ge 40, \quad 0.96 \le V(\mathbf{m}) \le 1.00, \quad
\mathbf{m}_{\text{lo}} \le \mathbf{m} \le \mathbf{m}_{\text{hi}},$$

where $\hat f$ is the surrogate's 28-day strength, `strength(model, m)`. The bounds go to the
optimiser directly. The other two constraints go into a penalty, so that every method of Tasks 4
and 5 minimises the same function (Task 6 handles them without one):

$$P(\mathbf{m}) = E(\mathbf{m}) + W\,\Bigl[\max\bigl(0,\ 40 - \hat f(\mathbf{m})\bigr)
+ 100 \max\bigl(0,\ 0.96 - V(\mathbf{m})\bigr) + 100 \max\bigl(0,\ V(\mathbf{m}) - 1.00\bigr)\Bigr], \qquad W = 1000 \text{ MJ per MPa}.$$

The factor 100 makes 0.01 m³ of volume as expensive as 1 MPa of strength. Use the constants
`TARGET`, `VOL_LO`, `VOL_HI` and `W`.

**Write**

- `energy(m)` and `volume(m)`: $E$ in MJ per m³ and $V$ in m³, for one mix (shape `(7,)`) or many
  (shape `(n, 7)`); both return an array of length $n$ (length 1 for one mix);
- `penalised(m, model)`: $P$ with $\hat f$ = `strength(model, m)`; a Python `float` for one mix, an
  array of length $n$ for many.

In [ ]:
def energy(m):
    ...


def volume(m):
    ...


def penalised(m, model):
    ...

In [ ]:
# Tests for Task 3 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_energy_and_volume_of_two_mixes():
    m_test = np.array([[300, 100, 50, 180, 6, 950, 750],
                       [500, 0, 0, 160, 10, 1000, 650]], float)
    assert np.allclose(energy(m_test), [1789.0, 2612.0]), \
        f"energy: expected [1789, 2612], got {energy(m_test)}"
    assert np.allclose(volume(m_test), [0.97133, 0.94272], atol=1e-4), \
        f"volume: expected [0.9713, 0.9427], got {volume(m_test)}"


test_energy_and_volume_of_two_mixes()


def test_penalised_with_a_fake_model():
    class Constant:
        # a fake model that always predicts 35 MPa
        def predict(self, X):
            return np.full(len(X), 35.0)


    m_test = np.array([[300, 100, 50, 180, 6, 950, 750],
                       [500, 0, 0, 160, 10, 1000, 650]], float)
    got = penalised(m_test, Constant())
    assert np.allclose(got, [6789.0, 9340.3], atol=0.1), \
        f"expected [6789.0, 9340.3], got {got}"


test_penalised_with_a_fake_model()


def test_one_mix_gives_a_float():
    class Constant:
        def predict(self, X):
            return np.full(len(X), 35.0)


    m = np.array([300, 100, 50, 180, 6, 950, 750], float)
    assert np.shape(energy(m)) == (1,) and np.shape(volume(m)) == (1,), \
        "energy and volume of one mix: arrays of length 1"
    assert isinstance(penalised(m, Constant()), float), "penalised of one mix: a float"


test_one_mix_gives_a_float()


print("all tests passed")

*Common ground: run this cell as it is, after your tests pass.*

In [ ]:
base = baseline[MIX].to_numpy(float)      # the baseline mix as an array, for Tasks 4-8
print("baseline mix, kg/m3:", base.round(1))

**Questions** (answer them in `practice_5_answers.md`, Task 3, with the numbers your code printed):

1. Compute the energy, the volume and the predicted 28-day strength (MLP) of the baseline mix, and compare the predicted strength with the measured one. Print `penalised(base, mlp)`: is the baseline feasible for the surrogate?
2. Take the baseline and cut the cement by 40 kg. Print energy, volume, predicted strength and the penalised value again. Which term of the penalty is active, and how big is it?
3. The penalty weight is 1000 MJ per MPa. In one or two sentences: what would go wrong with a weight of 10?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Common ground for Tasks 4–8: the call counter

Run this cell as it is. From here on **every optimiser is written by you**: Tasks 4, 5 and 6 build
grid search, random search, Nelder–Mead, differential evolution and SQP from their formulas.

To compare methods on the same budget we count the calls to the objective. `Counter` wraps a
function, counts every call and keeps every point and value. `f(x)` evaluates one point;
`f.batch(X)` evaluates an array of points of shape `(k, 7)` in one go and counts $k$ calls (a
grid, a random sample or a whole generation of DE). `best_so_far(budget)` is the curve to plot,
and `best_x(budget)` the best point found within the budget.

In [16]:
class Counter:
    # Wraps f(x) -> float. Counts the calls, keeps every x and every value.
    def __init__(self, fun):
        self.fun, self.calls, self.xs, self.values = fun, 0, [], []

    def __call__(self, x):
        v = float(self.fun(np.asarray(x, float)))
        self.calls += 1
        self.xs.append(np.array(x, float))
        self.values.append(v)
        return v

    def batch(self, X):
        # Many points at once, shape (k, n): one call of fun, k calls counted.
        X = np.atleast_2d(np.asarray(X, float))
        v = np.atleast_1d(np.asarray(self.fun(X), float))
        self.calls += len(X)
        self.xs.extend(X)
        self.values.extend(v.tolist())
        return v

    def best_so_far(self, budget):
        # k-th element: the smallest value among the first k + 1 calls,
        # cut at budget calls.
        return np.minimum.accumulate(self.values[:budget])

    def best_x(self, budget):
        # The point that gave the smallest value within the first budget calls.
        return self.xs[int(np.argmin(self.values[:budget]))]


BUDGET = 3 ** 7          # calls: a grid with 3 levels on each of the 7 ingredients

c = Counter(lambda x: (x[0] - 2) ** 2)
for v in [5, 1, 3]:
    c([v])
c.batch([[2.5], [2]])
print(f"Counter: {c.calls} calls, best so far {c.best_so_far(5)}, best x {c.best_x(5)}; "
      f"budget {BUDGET}")

Counter: 5 calls, best so far [9.   1.   1.   0.25], best x [2.5]; budget 2187


*Common ground for Task 4: the Nelder–Mead driver. It builds the first simplex, calls your
`nm_step` until the budget is used or the simplex has collapsed, and returns the best point. Run it
as it is.*

In [17]:
def nelder_mead(f, x0, lo, hi, max_calls, step=0.05, xatol=1e-2, fatol=1e-2):
    # Runs nm_step from x0 until max_calls calls of f, or until the simplex has
    # collapsed (all points within xatol of the best, all values within fatol).
    # Returns x, f(x), calls. The budget is checked between steps, so the last
    # step may overshoot it by a few calls; Counter.best_so_far(budget) cuts
    # the record at the budget.
    calls = 0

    def g(x):
        nonlocal calls
        calls += 1
        return f(x)

    # the first simplex: a step of 5 % of the range along each axis,
    # inwards at the upper bound
    n = len(x0)
    S = np.repeat(np.asarray(x0, float)[None, :], n + 1, axis=0)
    for i in range(n):
        delta = step * (hi[i] - lo[i])
        if x0[i] + delta <= hi[i]:
            S[i + 1, i] += delta
        else:
            S[i + 1, i] -= delta
    fS = np.array([g(x) for x in S])
    order = np.argsort(fS, kind="stable")
    S, fS = S[order], fS[order]

    while calls < max_calls:
        S, fS = nm_step(S, fS, g, lo, hi)
        small = np.abs(S[1:] - S[0]).max() <= xatol
        flat = np.abs(fS[1:] - fS[0]).max() <= fatol
        if small and flat:
            break
    return S[0], fS[0], calls

---

## Task 4. Grid, random search and Nelder–Mead, written by you  
*0.20 points: 0.12 for the code (tests), 0.08 for the answers*

**Grid search** puts $k$ levels on each of the $n$ variables and evaluates every
combination: $k^n$ points. With $k = 3$ and $n = 7$ that is 2187, the budget of this task.

**Random search** draws every point uniformly inside the box:
$\mathbf{x} = \mathbf{x}_{\text{lo}} + (\mathbf{x}_{\text{hi}} - \mathbf{x}_{\text{lo}}) \odot \mathbf{r}$,
$\mathbf{r} \sim U(0, 1)^n$.

**Nelder–Mead** (1965) keeps a simplex of $n + 1$ points (a triangle in two dimensions), sorted
from best to worst, $f_1 \le f_2 \le \dots \le f_{n+1}$. It moves the simplex using values only,
no slopes. One iteration, with $\bar{\mathbf{x}} = \frac{1}{n} \sum_{i=1}^{n} \mathbf{x}_i$ the
centroid of all points but the worst:

| move | new point | accepted (replaces $\mathbf{x}_{n+1}$) if |
|---|---|---|
| reflect | $\mathbf{x}_r = \bar{\mathbf{x}} + (\bar{\mathbf{x}} - \mathbf{x}_{n+1})$ | $f_1 \le f_r < f_n$ |
| expand, if $f_r < f_1$ | $\mathbf{x}_e = \bar{\mathbf{x}} + 2\,(\mathbf{x}_r - \bar{\mathbf{x}})$ | $f_e < f_r$; otherwise $\mathbf{x}_r$ is taken |
| contract outside, if $f_n \le f_r < f_{n+1}$ | $\mathbf{x}_c = \bar{\mathbf{x}} + \tfrac{1}{2}(\mathbf{x}_r - \bar{\mathbf{x}})$ | $f_c \le f_r$ |
| contract inside, if $f_r \ge f_{n+1}$ | $\mathbf{x}_c = \bar{\mathbf{x}} + \tfrac{1}{2}(\mathbf{x}_{n+1} - \bar{\mathbf{x}})$ | $f_c < f_{n+1}$ |
| shrink, if the contraction was rejected | $\mathbf{x}_i \leftarrow \mathbf{x}_1 + \tfrac{1}{2}(\mathbf{x}_i - \mathbf{x}_1)$, $i = 2, \dots, n + 1$ | always; the moved points are evaluated again |

Then sort the simplex again with `np.argsort(fS, kind="stable")`, so that ties are broken the same
way as in the tests. The worst corner is on the uphill side, so every move pushes the simplex away
from it; the step lengthens while the flips keep working and shortens when they fail. Every new
point is clipped into the box with `np.clip(x, lo, hi)` before it is evaluated, as scipy's
Nelder–Mead does with `bounds`. On a convex function the shrink never happens (the contracted
point lies between $\bar{\mathbf{x}}$ and a point no better than $\mathbf{x}_n$), so one test
uses a wavy function to reach it.

**Write**

- `grid_points(lo, hi, levels)`: every combination of `levels` equally spaced values from `lo[i]`
  to `hi[i]` per variable, shape `(levels**n, n)`, the first variable changing slowest, as in
  `itertools.product`;
- `random_points(lo, hi, n_points, rng)`: `n_points` uniform points inside the box, shape
  `(n_points, n)`, all randomness from the generator `rng`;
- `nm_step(S, fS, f, lo, hi)`: one Nelder–Mead iteration of the table. `S` has shape `(n + 1, n)`,
  sorted best first, `fS` holds their values; returns the new `S` and `fS`, sorted best first, and
  leaves its arguments unchanged. The driver `nelder_mead` above uses it;
- `runs`: a dict of three `Counter`s, each wrapping `lambda m: penalised(m, mlp)` with the budget
  `BUDGET`: `"grid"`, one batch of `grid_points(LO, HI, 3)`; `"random"`, one batch of `BUDGET` points
  of `random_points` with `np.random.default_rng(0)`; `"Nelder-Mead"`, `nelder_mead` from `base`,
  restarted from one point of `random_points` (its own generator, seeded with 0) whenever it stops,
  until the counter has used the budget.

In [ ]:
def grid_points(lo, hi, levels):
    ...


def random_points(lo, hi, n_points, rng):
    ...


def nm_step(S, fS, f, lo, hi):
    ...


runs = {"grid": ..., "random": ..., "Nelder-Mead": ...}

In [ ]:
# Tests for Task 4 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_grid_of_two_variables():
    got = np.asarray(grid_points(np.array([0, 10.]), np.array([1, 30.]), 3))
    assert got.shape == (9, 2), f"3 levels on 2 variables: shape (9, 2), got {got.shape}"
    want = list(itertools.product([0, 0.5, 1], [10, 20, 30]))
    assert np.allclose(got, want), "the order of itertools.product"
    assert np.shape(grid_points(LO, HI, 3)) == (2187, 7)


test_grid_of_two_variables()


def test_random_points_in_the_box():
    lo, hi = np.array([0, 10.]), np.array([1, 30.])
    r = np.asarray(random_points(lo, hi, 4000, np.random.default_rng(0)))
    assert r.shape == (4000, 2), f"shape (4000, 2), got {r.shape}"
    assert (r >= lo).all() and (r <= hi).all(), "every point inside the box"
    assert np.allclose(r.mean(0), [0.5, 20], atol=[0.03, 0.6]), \
        "uniform: the mean is the middle of the box"
    again = random_points(lo, hi, 4000, np.random.default_rng(0))
    other = random_points(lo, hi, 4000, np.random.default_rng(1))
    assert np.array_equal(r, again) and not np.allclose(r, other), \
        "the same seed gives the same points, another seed others"


test_random_points_in_the_box()


def test_nm_step_on_a_bowl():
    def run(fun, S, steps, lo=np.array([-5, -5.]), hi=np.array([5, 5.])):
        S = np.asarray(S, float)
        fS = np.array([fun(x) for x in S])
        order = np.argsort(fS, kind="stable")
        S, fS = S[order], fS[order]
        for _ in range(steps):
            S, fS = nm_step(S, fS, fun, lo, hi)
        return S

    # three steps: an expansion, a reflection and both contractions
    bowl = lambda x: (x[0] - 1) ** 2 + 10 * (x[1] - 2) ** 2
    start = [[0, 0], [0.5, 0], [0, 0.5]]
    S3 = run(bowl, start, 3)
    want = [[0.875, 2.125], [0.125, 1.875], [0.75, 0.75]]
    assert np.allclose(S3, want), \
        f"after 3 steps expected {want}, got {np.round(S3, 4).tolist()}"
    assert np.allclose(run(bowl, start, 40)[0], [1, 2], atol=1e-4), \
        "40 steps reach the minimum (1, 2)"


test_nm_step_on_a_bowl()


def test_nm_step_shrinks_on_a_wave():
    def run(fun, S, steps, lo=np.array([-5, -5.]), hi=np.array([5, 5.])):
        S = np.asarray(S, float)
        fS = np.array([fun(x) for x in S])
        order = np.argsort(fS, kind="stable")
        S, fS = S[order], fS[order]
        for _ in range(steps):
            S, fS = nm_step(S, fS, fun, lo, hi)
        return S

    # not convex: one step is a shrink
    wave = lambda x: np.sin(3 * x[0]) + np.cos(3 * x[1])
    Sw = run(wave, [[0, -0.7], [0.5, -0.8], [-0.2, 0]], 1)
    assert np.allclose(Sw, [[0, -0.7], [0.25, -0.75], [-0.1, -0.35]]), \
        f"a shrink towards (0, -0.7) expected, got {np.round(Sw, 4).tolist()}"


test_nm_step_shrinks_on_a_wave()


def test_nm_step_on_a_kink():
    def run(fun, S, steps, lo=np.array([-5, -5.]), hi=np.array([5, 5.])):
        S = np.asarray(S, float)
        fS = np.array([fun(x) for x in S])
        order = np.argsort(fS, kind="stable")
        S, fS = S[order], fS[order]
        for _ in range(steps):
            S, fS = nm_step(S, fS, fun, lo, hi)
        return S

    kink = lambda x: abs(x[0]) + abs(x[1])
    best = run(kink, [[0.1, 0.1], [0.3, -0.2], [-0.25, 0.3]], 6)[0]
    assert np.allclose(best, [-0.012891, 0.033203], atol=1e-6), \
        f"after 6 steps expected (-0.0129, 0.0332), got {best}"


test_nm_step_on_a_kink()


def test_nelder_mead_finds_the_bowl_minimum():
    bowl = lambda x: (x[0] - 1) ** 2 + 10 * (x[1] - 2) ** 2
    lo, hi = np.array([-5, -5.]), np.array([5, 5.])
    x, fx, calls = nelder_mead(bowl, np.array([3., -3.]), lo, hi, max_calls=400)
    assert np.allclose(x, [1, 2], atol=0.05), \
        f"the driver with your nm_step from (3, -3): expected (1, 2), got {x}"


test_nelder_mead_finds_the_bowl_minimum()


def test_runs_use_the_budget():
    assert {"grid", "random", "Nelder-Mead"} <= set(runs), "runs: grid, random, Nelder-Mead"
    assert all(isinstance(f, Counter) for f in runs.values()), \
        "every entry of runs is a Counter"
    assert runs["grid"].calls == BUDGET and runs["random"].calls == BUDGET, \
        "grid and random use exactly BUDGET calls"
    assert BUDGET <= runs["Nelder-Mead"].calls <= BUDGET + 20, \
        "Nelder-Mead restarts until the budget is used"


test_runs_use_the_budget()


print("all tests passed")

**Questions** (answer them in `practice_5_answers.md`, Task 4, with the numbers your code printed):

1. Print a table: for each method the best mix found within the budget, its energy, predicted strength and volume, and the number of calls it used. Plot the best penalised value so far against the number of calls for the three methods, with the baseline energy as a horizontal line (Task 5 adds differential evolution to both).
2. Which methods beat the baseline, by how many MJ and percent? Why does the grid do so badly with 2187 points?
3. How many calls did the first Nelder–Mead run take, and did the restarts add anything?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

*Common ground for Task 5: the driver of differential evolution. It makes every random choice and
hands it to your two functions; with the same seed everybody gets the same run. Run it as it is.*

In [21]:
def differential_evolution(f_batch, lo, hi, pop, generations, seed,
                           F=0.5, CR=0.7, tol=None):
    # f_batch maps an array (pop, n) to (pop,). Every random choice is made here.
    # Costs pop * (generations + 1) calls, fewer if tol stops it early.
    # Returns the best point, its value and the number of generations run.
    rng = np.random.default_rng(seed)
    n = len(lo)
    P = lo + (hi - lo) * rng.random((pop, n))
    fP = f_batch(P)
    for gen in range(1, generations + 1):
        # three different partners for every member, none of them the member itself
        r = np.array([rng.choice(np.delete(np.arange(pop), i), 3, replace=False)
                      for i in range(pop)])

        # crossover mask; j_rand makes sure every trial takes at least one coordinate
        cross = rng.random((pop, n)) < CR
        j_rand = rng.integers(0, n, pop)
        cross[np.arange(pop), j_rand] = True

        U = de_trial(P, r, F, cross, lo, hi)
        P, fP = de_select(P, fP, U, f_batch(U))
        if tol is not None and np.std(fP) <= tol * abs(np.mean(fP)):
            break
    i = int(np.argmin(fP))
    return P[i], fP[i], gen

---

## Task 5. Differential evolution, written by you, and a forest  
*0.20 points: 0.12 for the code (tests), 0.08 for the answers*

**Differential evolution** (Storn and Price, 1997) keeps a population of $N$ points and
improves all of them at once. One generation, for every member $\mathbf{x}_i$:

| step | rule |
|---|---|
| mutation | $\mathbf{v}_i = \mathbf{a} + F\,(\mathbf{b} - \mathbf{c})$, with $\mathbf{a}, \mathbf{b}, \mathbf{c}$ three other members, all different |
| crossover | $u_{ij} = v_{ij}$ if $\rho_{ij} < CR$ or $j = j_\text{rand}$, otherwise $x_{ij}$; $\rho_{ij} \sim U(0, 1)$ |
| clip | $\mathbf{u}_i$ is clipped into the box |
| selection | $\mathbf{u}_i$ replaces $\mathbf{x}_i$ if $f(\mathbf{u}_i) \le f(\mathbf{x}_i)$ |

The difference of two members sets the length and the direction of the step. While the population
is spread out the steps are long; once it has gathered in a valley they are short, and nobody
tunes a step size by hand. The coordinate $j_\text{rand}$ always comes from the mutant, so that the
trial differs from $\mathbf{x}_i$. Selection never makes a member worse. This variant is called
rand/1/bin; we use $F = 0.5$ and $CR = 0.7$. scipy's default, best/1/bin, builds every mutant
around the best member: it converges faster and explores less.

The driver above makes every random choice and hands it over: `r` holds the indices of
$\mathbf{a}, \mathbf{b}, \mathbf{c}$ for every member, shape `(N, 3)`; `cross` is the boolean mask
of the crossover with $j_\text{rand}$ already set, shape `(N, n)`. A generation is evaluated in one
call of `f_batch`, so it costs $N$ calls.

**Write**

- `de_trial(P, r, F, cross, lo, hi)`: the trials of the whole population, shape `(N, n)`:
  mutation, crossover and clip (fancy indexing `P[r[:, 0]]` and `np.where` do it without a loop);
- `de_select(P, fP, U, fU)`: the new population and its values, `(P, fP)`; a trial replaces its
  member where it is at least as good. The arguments stay unchanged;
- `runs["DE"]`: a `Counter` of `lambda m: penalised(m, mlp)` used by
  `differential_evolution(f.batch, LO, HI, 70, 30, seed=0)`, which makes $70 + 30 \cdot 70 = 2170$
  calls;
- `runs_rf`: a dict with `"Nelder-Mead"` and `"DE"`, the same two runs as `runs["Nelder-Mead"]` and
  `runs["DE"]` (same start, restarts, seeds and budgets) on the **forest**,
  `lambda m: penalised(m, forest)`;
- `first_rf`: the number of calls the *first* Nelder–Mead run on the forest used before it stopped.

The forest predicts slowly: its Nelder–Mead run takes about a minute.

In [ ]:
def de_trial(P, r, F, cross, lo, hi):
    ...


def de_select(P, fP, U, fU):
    ...


runs["DE"] = ...
runs_rf = {"Nelder-Mead": ..., "DE": ...}
first_rf = ...

In [ ]:
# Tests for Task 5 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_de_trial_on_five_members():
    P = np.array([[0, 0], [1, 0], [0, 1], [1, 1], [2, 2]], float)
    r = np.array([[1, 2, 3], [2, 3, 4], [3, 4, 0], [4, 0, 1], [0, 1, 2]])
    cross = np.array([[1, 0], [0, 1], [1, 1], [0, 0], [1, 0]], bool)
    lo, hi = np.array([0, 0.]), np.array([2, 1.5])
    U = np.asarray(de_trial(P, r, 0.5, cross, lo, hi))
    want = [[0.5, 0], [1, 0.5], [2, 1.5], [1, 1], [0.5, 1.5]]
    assert np.allclose(U, want), f"expected {want}, got {U.tolist()}"


test_de_trial_on_five_members()


def test_de_select_keeps_the_better():
    P = np.array([[0, 0], [1, 0], [0, 1], [1, 1], [2, 2]], float)
    U = np.array([[0.5, 0], [1, 0.5], [2, 1.5], [1, 1], [0.5, 1.5]])
    fP = np.array([3, 2, 5, 1, 4.])
    fU = np.array([1, 2, 6, 0.5, 7.])
    P2, f2 = de_select(P, fP, U, fU)
    assert np.allclose(P2, [[0.5, 0], [1, 0.5], [0, 1], [1, 1], [2, 2]]), \
        "a trial wins where it is at least as good (a tie goes to the trial)"
    assert np.allclose(f2, [1, 2, 5, 0.5, 4]), "the values follow the points"
    unchanged_P = np.allclose(P, [[0, 0], [1, 0], [0, 1], [1, 1], [2, 2]])
    unchanged_fP = np.allclose(fP, [3, 2, 5, 1, 4])
    assert unchanged_P and unchanged_fP, "de_select must not change its arguments"


test_de_select_keeps_the_better()


def test_de_finds_the_bowl_minimum():
    bowl_batch = lambda X: (X[:, 0] - 1) ** 2 + 10 * (X[:, 1] - 2) ** 2
    lo, hi = np.array([-5, -5.]), np.array([5, 5.])
    x, fx, gens = differential_evolution(bowl_batch, lo, hi, 20, 100, seed=0)
    assert np.allclose(x, [1, 2], atol=1e-3), \
        f"the driver with your operators: expected (1, 2), got {x}"


test_de_finds_the_bowl_minimum()


def test_forest_runs_reported():
    assert isinstance(runs["DE"], Counter) and runs["DE"].calls == 70 * 31, \
        "runs['DE']: a Counter with 2170 calls"
    assert {"Nelder-Mead", "DE"} <= set(runs_rf)
    assert all(isinstance(f, Counter) for f in runs_rf.values())
    assert isinstance(first_rf, (int, np.integer)), "first_rf: calls of the first run"
    assert 0 < first_rf <= runs_rf["Nelder-Mead"].calls, "first_rf: calls of the first run"


test_forest_runs_reported()


print("all tests passed")

**Questions** (answer them in `practice_5_answers.md`, Task 5, with the numbers your code printed):

1. Add DE to the table and the figure of Task 4. How does it compare with Nelder–Mead on the MLP?
2. On the forest: how many calls does the *first* Nelder–Mead run take before it stops, and how far does it get from the baseline? What does DE find on the forest? Read the penalised value as well as the energy.
3. Plot the predicted 28-day strength of both surrogates along one line: the baseline mix with the cement changed from 100 to 400 kg, everything else fixed. Explain from the plot why Nelder–Mead stalls on the forest and not on the MLP. Which kind of surrogate does a local optimiser need?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

*Common ground for Task 6: SQP. `solve_qp` solves the quadratic subproblem exactly; `sqp` is the
driver, and it calls your `fd_gradient` and `bfgs_update`. Run it as it is.*

In [25]:
from scipy.optimize import nnls      # the only library call: non-negative least squares


def solve_qp(B, g, G, b):
    # min g.d + 1/2 d'Bd subject to G d >= b, B positive definite.
    # Returns d and the multipliers (one per row of G),
    # or (None, None) if the constraints contradict each other.
    # With B = L L' and u = L'd + L^-1 g this is:
    # the shortest u with E u >= e (Lawson-Hanson).
    L = np.linalg.cholesky(B)
    Lg = np.linalg.solve(L, g)
    E = np.linalg.solve(L, G.T).T
    e = b + E @ Lg
    n = len(g)

    A = np.vstack([E.T, e[None, :]])
    target = np.r_[np.zeros(n), 1.0]
    w, _ = nnls(A, target, maxiter=50 * A.shape[1])
    res = A @ w - target
    if np.linalg.norm(res) < 1e-12 or res[-1] >= 0:
        return None, None

    u = -res[:n] / res[-1]
    return np.linalg.solve(L.T, u - Lg), w / (-res[-1])


def sqp(obj, cons, x0, lo, hi, max_iter=100, h=1e-6):
    # min obj(x) subject to cons(x) >= 0 and lo <= x <= hi.
    # obj maps (k, n) -> (k,), cons maps (k, n) -> (k, m).
    # Works in z = (x - lo) / (hi - lo). Counts the rows given to cons.
    # Returns a dict: x, f, c (constraints at x), lam (their multipliers),
    # iters, calls, path, message.
    span, n = hi - lo, len(x0)
    calls = 0

    def fo(Z):
        return np.asarray(obj(lo + span * np.atleast_2d(Z)), float)

    def fc(Z):
        nonlocal calls
        Z = np.atleast_2d(Z)
        calls += len(Z)
        return np.asarray(cons(lo + span * Z), float)

    z = (np.asarray(x0, float) - lo) / span
    B, rho = np.eye(n), 0.0

    # 1. slopes
    gf, f0 = fd_gradient(fo, z, h)
    J, c0 = fd_gradient(fc, z, h)
    lam = np.zeros(len(c0))
    path, message = [lo + span * z], "iteration limit"

    for it in range(1, max_iter + 1):
        # 2. the quadratic subproblem, with the box as extra rows
        G = np.vstack([J, np.eye(n), -np.eye(n)])
        d, mult = solve_qp(B, gf, G, np.concatenate([-c0, -z, z - 1]))
        if d is None:
            message = "linearised constraints incompatible"
            break
        lam = mult[:len(c0)]

        # 3. step length on the merit function phi = f + rho * violation
        rho = max(lam.max(), 0.5 * (rho + lam.max()))
        viol0 = np.maximum(0, -c0).sum()
        phi0 = f0 + rho * viol0
        dphi = gf @ d - rho * viol0
        if abs(dphi) < 1e-4 * max(1, abs(f0)) and viol0 < 1e-6:
            message = "converged"
            break
        alpha = 1.0
        while True:
            zt = z + alpha * d
            ft, ct = fo(zt)[0], fc(zt)[0]
            phi = ft + rho * np.maximum(0, -ct).sum()
            if phi <= phi0 + 1e-4 * alpha * dphi or alpha < 1e-3:
                break
            alpha /= 2
        if alpha < 1e-3:
            message = "line search failed"
            break

        # 4. curvature: BFGS on the gradient of the Lagrangian
        gf_new, f_new = fd_gradient(fo, zt, h)
        J_new, c_new = fd_gradient(fc, zt, h)
        s = zt - z
        y = (gf_new - J_new.T @ lam) - (gf - J.T @ lam)
        B = bfgs_update(B, s, y)

        # accept the step
        z, gf, f0, J, c0 = zt, gf_new, f_new, J_new, c_new
        path.append(lo + span * z)

    return {"x": lo + span * z, "f": f0, "c": c0, "lam": lam, "iters": it,
            "calls": calls, "path": np.array(path), "message": message}

---

## Task 6. SQP, written by you: the constraints without a penalty  
*0.20 points: 0.12 for the code (tests), 0.08 for the answers*

**SQP, sequential quadratic programming**, is what scipy's SLSQP does (Kraft, 1988).
It needs slopes, and in return it handles constraints exactly, without a penalty. The problem in
standard form:

$$\min_{\mathbf{x}} f(\mathbf{x}) \quad \text{subject to} \quad \mathbf{c}(\mathbf{x}) \ge \mathbf{0}, \qquad \mathbf{x}_{\text{lo}} \le \mathbf{x} \le \mathbf{x}_{\text{hi}}.$$

Here $f = E$, and $\mathbf{c}$ has three components: $\hat f(\mathbf{m}) - 40$,
$V(\mathbf{m}) - 0.96$ and $1.00 - V(\mathbf{m})$.

**What holds at the optimum.** With the Lagrangian
$\mathcal{L}(\mathbf{x}, \boldsymbol{\lambda}) = f(\mathbf{x}) - \boldsymbol{\lambda}^\top \mathbf{c}(\mathbf{x})$,
the Karush–Kuhn–Tucker (KKT) conditions for an optimum away from the bounds are

$$\nabla f(\mathbf{x}^*) = \sum_j \lambda_j \nabla c_j(\mathbf{x}^*), \qquad c_j(\mathbf{x}^*) \ge 0, \qquad \lambda_j \ge 0,
\qquad \lambda_j\, c_j(\mathbf{x}^*) = 0.$$

Every constraint is either inactive ($c_j > 0$, then $\lambda_j = 0$) or active ($c_j = 0$, the
optimum lies on it). At the optimum the downhill direction of $f$ is blocked by the active
constraints, and $\lambda_j$ says how hard constraint $j$ pushes back. It is the price of the
constraint: how much $f$ would drop if the constraint were relaxed by one unit. For the strength
it is in MJ per MPa.

**One iteration** at $\mathbf{x}_k$, with a matrix $\mathbf{B}_k$ that estimates the curvature of
$\mathcal{L}$, starting from $\mathbf{B}_0 = \mathbf{I}$:

| step | what the driver does |
|---|---|
| 1. slopes | forward differences, $n + 1$ evaluations: $\partial f / \partial x_i \approx \bigl(f(\mathbf{x}_k + h\,\mathbf{e}_i) - f(\mathbf{x}_k)\bigr) / h$, and the same for every component of $\mathbf{c}$, which gives the Jacobian $\mathbf{J}_k$, one row per constraint |
| 2. subproblem | $\min_{\mathbf{d}}\ \nabla f_k^\top \mathbf{d} + \tfrac{1}{2} \mathbf{d}^\top \mathbf{B}_k \mathbf{d}$ subject to $\mathbf{c}_k + \mathbf{J}_k \mathbf{d} \ge \mathbf{0}$ and the box, solved exactly by `solve_qp`, which also returns $\boldsymbol{\lambda}$ |
| 3. step length | $\alpha = 1, \tfrac{1}{2}, \tfrac{1}{4}, \dots$ until the merit function $\varphi = f + \rho \sum_j \max(0, -c_j)$ drops enough; $\rho \ge \max_j \lambda_j$, below that breaking a constraint would pay |
| 4. curvature | the damped BFGS update below, with $\mathbf{s} = \mathbf{x}_{k+1} - \mathbf{x}_k$ and $\mathbf{y} = \nabla_{\mathbf{x}} \mathcal{L}(\mathbf{x}_{k+1}, \boldsymbol{\lambda}) - \nabla_{\mathbf{x}} \mathcal{L}(\mathbf{x}_k, \boldsymbol{\lambda})$ |

$$\mathbf{B}_{k+1} = \mathbf{B}_k - \frac{\mathbf{B}_k \mathbf{s}\, \mathbf{s}^\top \mathbf{B}_k}{\mathbf{s}^\top \mathbf{B}_k \mathbf{s}} + \frac{\mathbf{r}\, \mathbf{r}^\top}{\mathbf{s}^\top \mathbf{r}},
\qquad \mathbf{r} = \theta \mathbf{y} + (1 - \theta) \mathbf{B}_k \mathbf{s},
\qquad \theta = \begin{cases} 1 & \text{if } \mathbf{s}^\top \mathbf{y} \ge 0.2\, \mathbf{s}^\top \mathbf{B}_k \mathbf{s}, \\[2pt]
\dfrac{0.8\, \mathbf{s}^\top \mathbf{B}_k \mathbf{s}}{\mathbf{s}^\top \mathbf{B}_k \mathbf{s} - \mathbf{s}^\top \mathbf{y}} & \text{otherwise.} \end{cases}$$

With $\theta = 1$ this is plain BFGS, and then $\mathbf{B}_{k+1} \mathbf{s} = \mathbf{y}$: the new
matrix reproduces the change of slope it has just seen, the way a secant reproduces two points of
a curve. Powell's damping, $\theta < 1$, keeps $\mathbf{B}$ positive definite when the curvature
along $\mathbf{s}$ is negative or tiny. Without it the parabola of step 2 could turn upside down
and have no minimum.

The driver works in scaled variables $\mathbf{z} = (\mathbf{x} - \mathbf{x}_{\text{lo}}) / (\mathbf{x}_{\text{hi}} - \mathbf{x}_{\text{lo}})$,
from 0 to 1, so that a kilogram of superplasticizer and a kilogram of gravel are not the same
step; $h = 10^{-6}$ in those units. `solve_qp` is the "least squares" of SLSQP: with
$\mathbf{B} = \mathbf{L} \mathbf{L}^\top$ the subproblem becomes the search for the shortest vector
that satisfies linear inequalities, which reduces to non-negative least squares (Lawson and
Hanson, 1974).

**Write**

- `fd_gradient(fun, x, h)`: `fun` maps an array of points `(k, n)` to `(k,)` or to `(k, m)`. Call it
  **once**, with the $n + 1$ points $\mathbf{x}$ and $\mathbf{x} + h\,\mathbf{e}_i$; return the
  forward-difference gradient `(n,)` or Jacobian `(m, n)`, and the value of `fun` at `x` (a float
  or an array `(m,)`);
- `bfgs_update(B, s, y)`: $\mathbf{B}_{k+1}$ of the damped update, shape `(n, n)`;
- `make_cons(model)`: a function `cons(m)` that maps mixes `(k, 7)` to `(k, 3)`, the columns
  `strength(model, m) - TARGET`, `volume(m) - VOL_LO` and `VOL_HI - volume(m)`;
- `res_sqp` and `res_sqp_rf`: the dicts returned by `sqp(energy, make_cons(model), base, LO, HI)`
  with the MLP and with the forest.

In [ ]:
def fd_gradient(fun, x, h):
    ...


def bfgs_update(B, s, y):
    ...


def make_cons(model):
    ...


res_sqp = ...
res_sqp_rf = ...

In [ ]:
# Tests for Task 6 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_gradient_of_a_quadratic():
    quadratic = lambda X: X[:, 0] ** 2 + 3 * X[:, 0] * X[:, 1]
    g, v = fd_gradient(quadratic, np.array([1., 2.]), 1e-6)
    assert np.allclose(g, [8, 3], atol=1e-4), \
        f"gradient of x0^2 + 3 x0 x1 at (1, 2): expected [8, 3], got {g}"
    assert np.isclose(v, 7), f"the value at (1, 2) is 7, got {v}"


test_gradient_of_a_quadratic()


def test_jacobian_of_two_functions():
    two_functions = lambda X: np.column_stack([X[:, 0] * X[:, 1], np.sin(X[:, 0])])
    J, v = fd_gradient(two_functions, np.array([1., 2.]), 1e-6)
    J = np.asarray(J)
    assert J.shape == (2, 2), f"Jacobian: one row per function, shape (2, 2), got {J.shape}"
    assert np.allclose(J, [[2, 1], [0.540302, 0]], atol=1e-4), \
        f"expected [[2, 1], [0.5403, 0]], got {np.round(J, 4).tolist()}"
    assert np.allclose(v, [2, np.sin(1)]), "the values at x"


test_jacobian_of_two_functions()


def test_one_call_with_n_plus_one_points():
    seen = []


    def fun(X):
        seen.append(np.array(X))
        return X.sum(1)


    fd_gradient(fun, np.array([1., 2., 3.]), 1e-6)
    assert len(seen) == 1, f"fun must be called once, it was called {len(seen)} times"
    assert seen[0].shape == (4, 3), f"with n + 1 = 4 points, got shape {seen[0].shape}"


test_one_call_with_n_plus_one_points()


def test_bfgs_known_values():
    B1 = bfgs_update(np.eye(2), np.array([1, 0.5]), np.array([2, 0.4]))
    B2 = bfgs_update(np.eye(2), np.array([1, 0.5]), np.array([-1, 0.2]))
    plain = [[2.018182, -0.036364], [-0.036364, 0.872727]]
    damped = [[0.21947, -0.299405], [-0.299405, 1.31974]]
    assert np.allclose(B1, plain, atol=1e-6), f"plain BFGS: got {np.round(B1, 4).tolist()}"
    assert np.allclose(B2, damped, atol=1e-5), f"damped BFGS: got {np.round(B2, 4).tolist()}"


test_bfgs_known_values()


def test_bfgs_secant_and_positive_definite():
    s = np.array([1, 0.5])
    B1 = bfgs_update(np.eye(2), s, np.array([2, 0.4]))
    assert np.allclose(B1 @ s, [2, 0.4]), "without damping the secant condition B s = y holds"
    B2 = bfgs_update(np.eye(2), s, np.array([-1, 0.2]))
    assert np.allclose(B2, B2.T) and np.all(np.linalg.eigvalsh(B2) > 0), \
        "with negative curvature the damped B stays symmetric positive definite"


test_bfgs_secant_and_positive_definite()


def test_constraints_with_a_fake_model():
    class Constant:
        def predict(self, X):
            return np.full(len(X), 35.0)


    m = np.array([[300, 100, 50, 180, 6, 950, 750],
                  [500, 0, 0, 160, 10, 1000, 650]], float)
    c = np.asarray(make_cons(Constant())(m))
    assert c.shape == (2, 3), f"cons maps (k, 7) to (k, 3), got shape {c.shape}"
    want = [[-5, 0.01133, 0.02867], [-5, -0.01728, 0.05728]]
    assert np.allclose(c, want, atol=1e-4), f"got {np.round(c, 4).tolist()}"


test_constraints_with_a_fake_model()


def test_sqp_on_the_disc():
    # min x0 + x1 on the unit disc: the answer is (-1/sqrt2, -1/sqrt2) with lambda = 1/sqrt2
    total = lambda X: X.sum(1)
    inside_disc = lambda X: (1 - (X ** 2).sum(1))[:, None]
    lo, hi = np.array([-2, -2.]), np.array([2, 2.])
    res = sqp(total, inside_disc, np.array([0.5, 0.2]), lo, hi)
    assert np.allclose(res["x"], -np.sqrt(0.5), atol=1e-4), \
        f"expected (-0.7071, -0.7071), got {res['x']}"
    assert np.allclose(res["lam"], np.sqrt(0.5), atol=1e-3), \
        f"multiplier 0.7071, got {res['lam']}"


test_sqp_on_the_disc()


def test_sqp_results_reported():
    for name, res in {"res_sqp": res_sqp, "res_sqp_rf": res_sqp_rf}.items():
        assert isinstance(res, dict) and {"x", "lam", "message", "calls"} <= set(res), \
            f"{name}: the dict returned by sqp"


test_sqp_results_reported()


print("all tests passed")

**Questions** (answer them in `practice_5_answers.md`, Task 6, with the numbers your code printed):

1. Report the SQP optimum on the MLP: the mix, its energy, strength and volume, the number of iterations and of calls to the model, and the stop message. Add SQP to the table of Task 5.
2. Which constraints are active at the optimum? Read the multipliers: what is one MPa of the strength requirement worth, in MJ per m³? Compare it with the penalty weight $W = 1000$ of Task 3, and with the weight of 10 that Task 3 warned about.
3. What happens to SQP on the **forest**, and why? Use the plot of Task 5.

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

*Common ground for Task 7: four more networks with seeds 1–4, the scaler of the ingredients and
the real mixes in scaled units. Run it as it is.*

In [29]:
# five MLPs, seeds 0-4
ensemble = [mlp] + [make_mlp(seed).fit(X_all, y_all) for seed in range(1, 5)]
# each ingredient divided by its standard deviation
scale = StandardScaler().fit(concrete[MIX])
UNIQ = concrete[MIX].drop_duplicates()        # the distinct real mixes, kg/m3
REAL = scale.transform(UNIQ)                  # the same, in scaled units
print(f"{len(ensemble)} networks; {len(UNIQ)} distinct real mixes")

5 networks; 427 distinct real mixes


---

## Task 7. How far can the optimum be trusted?  
*0.20 points: 0.10 for the code (tests), 0.10 for the answers*

A surrogate's optimum lies where the surrogate is most optimistic, and that is often where it has
seen the least data. Three checks tell how far to trust it: a second model of a different kind
(the forest), the spread of several networks trained with different seeds, and the distance to
the nearest real mix.

The distance from a mix $\mathbf{a}$ to a set of mixes $\mathcal{B}$ is
$d(\mathbf{a}, \mathcal{B}) = \min_{\mathbf{b} \in \mathcal{B}} \lVert \mathbf{a} - \mathbf{b} \rVert$, measured in scaled units.
When the two sets are the same, a point must not count as its own neighbour: take the
second-smallest distance instead. $D_{95}$ is the 95th percentile of the distance from each real
mix to its nearest *other* real mix.

**The trusted objective.** With $\bar f$ and $s$ the mean and the standard deviation (`ddof=1`) of
the five networks of `ensemble`, the pessimistic strength $p = \bar f - 2s$, or
$p = \min(\bar f - 2s,\ \hat f_\text{forest})$ when the forest has to agree too:

$$T(\mathbf{m}) = E(\mathbf{m}) + W\,\Bigl[\max(0,\ \text{aim} - p) + 100 \max(0,\ 0.96 - V) + 100 \max(0,\ V - 1.00)
+ \max\bigl(0,\ d(\mathbf{m}) - D_{95}\bigr)\Bigr],$$

where $d(\mathbf{m})$ is the distance of the scaled mix to `REAL`. A unit of distance beyond
$D_{95}$ costs as much as a megapascal of strength.

**Write**

- `nearest_distance(A, B, exclude_self=False)`: for every row of `A`, the Euclidean distance to the
  closest row of `B`, shape `(len(A),)`; with `exclude_self=True` (then `A` is `B`) the distance to
  the closest *other* row. Numpy only; a table of all pairwise distances is fine for a few hundred
  mixes;
- `D95`: a float, $D_{95}$ of `REAL`;
- `trusted(aim=TARGET, with_forest=False)`: returns the function $T$, which maps mixes `(k, 7)` to
  `(k,)` (DE hands it a whole population);
- `best_ens` and `best_both`: arrays `(7,)`, the mixes found by
  `differential_evolution(trusted(...), LO, HI, 70, 300, seed=0, tol=1e-7)`, without and with the
  forest. Each run takes several seconds.

In [ ]:
def nearest_distance(A, B, exclude_self=False):
    ...


D95 = ...


def trusted(aim=TARGET, with_forest=False):
    ...


best_ens = ...
best_both = ...

In [ ]:
# Tests for Task 7 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_distance_to_the_nearest_point():
    A = np.array([[0., 0.], [3., 4.]])
    B = np.array([[1., 0.], [3., 0.], [6., 8.]])
    got = nearest_distance(A, B)
    assert np.allclose(got, [1, 4]), f"expected [1, 4], got {got}"


test_distance_to_the_nearest_point()


def test_excluding_the_point_itself():
    B = np.array([[0., 0.], [1., 0.], [5., 0.]])
    assert np.allclose(nearest_distance(B, B, exclude_self=True), [1, 1, 4]), \
        "the nearest other point"
    assert np.allclose(nearest_distance(B, B), [0, 0, 0]), \
        "without exclude_self every point is its own nearest"


test_excluding_the_point_itself()


def test_nearest_matches_sklearn():
    from sklearn.neighbors import NearestNeighbors
    rng = np.random.default_rng(1)
    A_demo = rng.normal(size=(50, 7))
    B_demo = rng.normal(size=(80, 7))
    nn = NearestNeighbors(n_neighbors=2).fit(B_demo)
    nearest_in_B, _ = nn.kneighbors(A_demo, 1)
    nearest_other, _ = nn.kneighbors(B_demo, 2)
    assert np.allclose(nearest_distance(A_demo, B_demo), nearest_in_B[:, 0])
    mine_other = nearest_distance(B_demo, B_demo, exclude_self=True)
    assert np.allclose(mine_other, nearest_other[:, 1])


test_nearest_matches_sklearn()


def test_trusted_gives_one_value_per_mix():
    mixes = UNIQ.to_numpy(float)[:5]
    v = np.asarray(trusted()(mixes))
    assert v.shape == (5,), f"trusted() maps (5, 7) to (5,), got shape {v.shape}"
    assert np.all(v >= mixes @ ENERGY - 1e-6), \
        "the trusted objective is the energy plus penalties"
    assert isinstance(D95, (float, np.floating)) and D95 > 0, "D95: a positive float"


test_trusted_gives_one_value_per_mix()


def test_trusted_optima_reported():
    assert np.shape(best_ens) == (7,) and np.shape(best_both) == (7,), \
        "best_ens and best_both: mixes of shape (7,)"


test_trusted_optima_reported()


print("all tests passed")

**Questions** (answer them in `practice_5_answers.md`, Task 7, with the numbers your code printed):

1. Take the best mix of Tasks 4–6, the lowest penalised value of the five methods, and check it three ways: what the forest predicts for it; the mean and standard deviation of the five networks; its distance to the nearest real mix against $D_{95}$. How many checks does it pass?
2. Run the same checks on `best_ens`. What does the forest say now, and why is the agreement of five networks weak evidence?
3. Run the checks on `best_both`. How much of the saving over the baseline is left, and which ingredients did the optimiser change?
4. Print the nearest real mix to `best_both`: its ingredients and its *measured* strengths. Which of the optima would you dare to cast?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 8. The safety margin and the recommendation  
*0.10 points: 0.04 for the code (tests), 0.06 for the answers*

A surrogate is wrong by about `RMSE_MLP` on a new mix, and the plant's own batches scatter too. The
usual answer is to aim higher than the requirement: ask for $40 + z \cdot \text{RMSE}$ MPa. For a
fair comparison the baseline moves too: at each aim, the least-energy 28-day test in `d28` whose
**measured** strength reaches the same aim.

**Write** `margin_table`: a `pd.DataFrame` indexed by $z$ = 0, 0.5, 1, 1.645, with the columns

| column | what it holds |
|---|---|
| `"aim, MPa"` | $40 + z \cdot$ `RMSE_MLP` |
| `"optimum, MJ/m3"` | the energy of the mix found by `differential_evolution(trusted(aim, with_forest=True), LO, HI, 70, 300, seed=0, tol=1e-7)` |
| `"best real mix, MJ/m3"` | the least energy of a test in `d28` with `csMPa` at least the aim |
| `"difference, %"` | $100\,(\text{optimum} / \text{best real} - 1)$ |

The four runs take a minute or two.

In [ ]:
margin_table = ...

In [ ]:
# Tests for Task 8 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_margin_table_shape():
    assert isinstance(margin_table, pd.DataFrame), "margin_table: a DataFrame"
    z = margin_table.index.to_numpy(float)
    assert np.allclose(z, [0, 0.5, 1, 1.645]), "index: z = 0, 0.5, 1, 1.645"
    cols = ["aim, MPa", "optimum, MJ/m3", "best real mix, MJ/m3", "difference, %"]
    assert set(cols) <= set(margin_table.columns), f"columns {cols}"
    assert np.allclose(margin_table["aim, MPa"], TARGET + z * RMSE_MLP), \
        "aim = 40 + z * RMSE_MLP"


test_margin_table_shape()


print("all tests passed")

**Questions** (answer them in `practice_5_answers.md`, Task 8, with the numbers your code printed):

1. Print the table and plot both energies against the aim. Where does the optimiser save against the best real mix at the same aim, and where not?
2. Your recommendation to the plant, in five to eight sentences: which mix goes to a **trial batch** first and with what aim; what the lab must measure; what you would do with the result (In-class work 9, Part 6.1, does the same with a dryer).
3. Your conclusions in ten to fifteen lines: which optimiser you would use for this kind of problem and why, with the numbers of Tasks 4–6; what the surrogate's optimum of Tasks 4–6 is worth, and what the trusted one of Task 7 is worth; what single experiment would teach you the most about whether the saving is real.

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers